# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_414/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

... (5 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [16]:
CANDIDATES = 20


def tokenize(text):
    return re.findall(r"\w+", text.lower())


bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)


def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)


def bm25_search(query, k):
    return bm25.invoke(query)[:k]


def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()


for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)


my_queries = [
    # 1. Точний термін / номер / абревіатура (перемагає BM25)
    "Що таке ЄКТС?",
    # 2. Перефразування без точних слів із документа (перемагає vector_search)
    "Чи дозволено в гуртожитку вмикати голосну музику вночі?",
]

for q in my_queries:
    print("=" * 60)
    show(vector_search, q)
    show(bm25_search, q)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [8]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



In [17]:
c = 60
rank_vector = 2
rank_bm25 = 5

rrf_score = 1 / (c + rank_vector) + 1 / (c + rank_bm25)
print(f"Бал RRF: {rrf_score:.6f} (точний дріб: {rrf_score})")

Бал RRF: 0.031514 (точний дріб: 0.0315136476426799)


## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [9]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [10]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


In [18]:
BENCHMARK += [
    # 1. Власне питання з відповіддю в документах (з Положення про користування гуртожитками [348], п. 5.2)
    ("Чи можна платити за гуртожиток щомісяця?", ["348"], r"авансовий платіж|щомісяц", "Ні, студент вносить авансовий платіж за 1 семестр до 1 вересня, а за 2 семестр — до 1 лютого."),
    # 2. Перефразування наявного питання ("У який час гості можуть перебувати в гуртожитку?")
    ("До котрої години дозволено залишатися відвідувачам у гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    # 3. Питання без відповіді в документах
    ("Який розмір штрафу за втрату студентського квитка?", [], None, None),
]

answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")
pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

Питань: 29, з них без відповіді в документах: 4


,метод,hit@1,hit@5,MRR
0,vector_search,0.64,0.88,0.76
1,bm25_search,0.48,0.64,0.56
2,hybrid_search,0.56,0.92,0.72
3,rerank_search,0.80,0.96,0.88


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [11]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [12]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [13]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [14]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


In [19]:
# Виводимо 5 відповідей із таблиці evaluation (зокрема ті, де суддя поставив False, та кілька True)
sample_df = pd.concat([
    evaluation[evaluation["правильно"] == False].head(3),
    evaluation[evaluation["правильно"] == True].head(3)
]).head(5)

ref_map = {q: ref for q, _, _, ref in BENCHMARK}

for idx, row in sample_df.iterrows():
    q = row["питання"]
    print(f"=== Приклад {idx + 1} ===")
    print(f"Питання:  {q}")
    print(f"Еталон:   {ref_map[q]}")
    print(f"Система:  {row['відповідь']}")
    print(f"Вердикт судді: {row['правильно']} | Цитування: {row['цитування']}\n")

=== Приклад 17 ===
Питання:  Для кого встановлюється стипендія ректора?
Еталон:   Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
Система:  Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].
Вердикт судді: False | Цитування: True

=== Приклад 23 ===
Питання:  Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
Еталон:   F2.
Система:  Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].
Вердикт судді: False | Цитування: False

=== Приклад 1 ===
Питання:  До якої дати потрібно оплатити гуртожиток за перший семестр?
Еталон:   До 1 вересня поточного року.
Система:  До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].
Вердикт судді: True | Цитування: True

=== Приклад 2

## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [15]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

# Питання:

1.Як довго виконувалися клітинки ноутбуку 03_rag_langchain_eval? Чи були проблеми з запуском, які?

2.Чим RecursiveCharacterTextSplitter (розділ 2) відрізняється від розбиття за реченнями з демо 2? Для чого задано параметр keep_separator="end"?

3.Чому у розділі 3 для FAISS обрано DistanceStrategy.MAX_INNER_PRODUCT? Чи можна було б так зробити, якби ембединги не були нормалізовані?

4.Знайдіть власне питання до документів, на якому BM25 знаходить правильний фрагмент вище, ніж семантичний пошук, і питання, на якому навпаки (розділ 4). Поясніть, чому так відбувається.

5.Обчисліть "вручну" (за допомогою Python коду без використання допоміжних діб) бал RRF для фрагмента, що стоїть на 2-й позиції у векторному пошуку й на 5-й позиції в BM25 (розділ 5). Чому RRF використовує лише позиції, а не самі оцінки методів пошуку?

6.Чому cross-encoder (розділ 6) застосовують лише до 20 кандидатів, а не до всіх фрагментів корпусу? Скільки проходів моделі знадобилося б на одне питання, щоб переранжувати весь корпус?

7.Додайте до бенчмарку (розділ 7) три власні питання: одне з відповіддю, одне, що перефразовує наявне питання, і одне без відповіді в документах. Як змінилися метрики ретриву?

8.Порівняйте метрики ретриву для чотирьох методів пошуку (розділ 7). Чому переранжування найсильніше впливає саме на hit@1?

9.У розділі 9 суддею виступає та сама модель, що генерує відповіді. Які проблеми це може спричинити? Перевірте вручну вердикти судді для пʼяти відповідей і порівняйте зі своїми.

10.Розгляньте неправильні відповіді в розділі 10 «Аналіз відмов». Для кожної визначте тип помилки: промах ретриву, помилка генерації, помилка судді чи неоднозначний еталон. Обґрунтуйте.

# Відповіді на питання:

1.Критичних помилок під час запуску не було, весь ноутбук відпрацював нормально. Але під час встановлення залежностей pip вивів попередження про конфлікт версій google-colab 1.0.0 вимагає requests==2.32.4, а встановлено requests 2.34.2, але це не критично і не завадило виконанню подальших клітинок.Всі клітинки ноутбука виконались за 9 хвилин.

2.На відміну від розбиття за реченнями з демо 2, яке використовувало регулярний вираз для пошуку меж речень і при перевищенні ліміту одразу переходило до посимвольного нарізання, RecursiveCharacterTextSplitter застосовує ієрархічний каскад роздільників ([". ", "; ", " ", ""]). Якщо фрагмент після поділу за реченнями виявляється довшим за 1000 символів, сплітер поступово знижує рівень деталізації: спочатку ділить за крапкою з комою, далі — за пробілами між словами, і лише в крайньому разі розриває окремі слова. Крім того, замість перекриття на одне останнє речення тут використовується фіксоване перекриття у 200 символів (chunk_overlap=200), у результаті чого корпус розбито на 385 фрагментів (проти 410 у попередній лабораторній роботі).

Параметр keep_separator="end" потрібен для того, щоб залишати символ роздільника (крапку або крапку з комою) наприкінці попереднього фрагмента, до якого він належить за змістом, а не переносити його на початок наступного. Це зберігає граматичну цілісність завершення попереднього уривка та запобігає появі зайвих розділових знаків на початку нового фрагмента.

3.Стратегію DistanceStrategy.MAX_INNER_PRODUCT обрано тому, що в налаштуваннях HuggingFaceEmbeddings для документів і запитів увімкнено L2-нормалізацію ("normalize_embeddings": True), яка приводить довжину кожного вектора до одиниці. Для нормалізованих векторів скалярний добуток математично збігається з косинусною подібністю, оскільки знаменник формули дорівнює одиниці. Використання максимального скалярного добутку у FAISS позбавляє систему потреби обчислювати норми векторів під час кожного пошуку, прискорює роботу та повертає оцінки подібності безпосередньо у діапазоні до 1 (наприклад, 0.60, 0.53, 0.50 на скріншоті).

Якби ембединги не були нормалізовані, використовувати MAX_INNER_PRODUCT для оцінки семантичної близькості було б некоректно. У такому разі скалярний добуток залежав би не лише від кута між векторами, а й від їхньої власної довжини: фрагменти з більшою магнітудою вектора штучно отримували б вищі оцінки та витісняли б релевантніші за змістом документи. Для ненормалізованих векторів довелося б або попередньо проводити нормалізацію, або використовувати стратегію DistanceStrategy.COSINE.

4.Питання, на якому BM25 знаходить правильний фрагмент вище, ніж семантичний пошук — «Що таке ЄКТС?» (або запит за точним рідкісним терміном чи номером пункту, наприклад «пункт 4.7»):
Алгоритм BM25 ставить цільовий фрагмент на першу позицію, вище за векторний пошук. Це відбувається тому, що BM25 базується на точному лексичному збігу слів і надає найвищу вагу рідкісним токенам, які зустрічаються лише в одному або кількох документах бази. Специфічна абревіатура чи точний номер пункту однозначно вказують на потрібний абзац, тоді як нейромережева модель ембедингів усереднює семантику короткого запиту і може підняти вище загальні описові фрагменти про систему навчання.

Питання, на якому семантичний пошук виграє у BM25 — «Чи дозволено в гуртожитку вмикати голосну музику вночі?»:
Як видно з результатів виконання, vector_search виводить у топ-3 фрагменти цільового документа [1219] (Правила внутрішнього розпорядку в студентських гуртожитках, 2-га та 3-тя позиції), де регламентується режим тиші в нічний час. Натомість bm25_search взагалі не знаходить документ [1219] у першій трійці, повертаючи нерелевантний фрагмент із документа [1199] (Кодекс академічної доброчесності) та загальні пункти з документа [348]. Причина полягає в тому, що в запиті використано синонімічний опис ситуації («голосна музика вночі»), тоді як у документі вжито формулювання про дотримання тиші з 23:00 до 7:00. Оскільки BM25 не розуміє синонімів і працює без лематизації (навіть різні відмінки одного слова вважаються різними токенами), за відсутності прямого збігу ключових слів він виявляється неефективним, на відміну від семантичного пошуку, який коректно вловлює зміст запиту.

5.Для обчислення підсумкового балу методом Reciprocal Rank Fusion (RRF) використовується формула $\sum \frac{1}{c + r}$, де $c = 60$ — константа згладжування, а $r$ — порядковий номер фрагмента у списку видачі. Для фрагмента на 2-й позиції у векторному пошуку та 5-й позиції у BM25 бал становить: $\frac{1}{60 + 2} + \frac{1}{60 + 5} = \frac{1}{62} + \frac{1}{65} \approx 0.031514$.

RRF оперує виключно порядковими позиціями (рангами), а не числовими оцінками релевантності, оскільки векторний пошук і BM25 використовують принципово різні та непорівнювані шкали вимірювання. Косинусна подібність змінюється в межах від $-1$ до $1$, тоді як оцінка BM25 не має верхньої межі і може сягати десятків одиниць залежно від частоти рідкісних слів у запиті. Пряме додавання таких оцінок призвело б до домінування BM25, а ручна нормалізація вимагала б складного калібрування під кожен набір даних. Перехід до рангів уніфікує внесок обох методів без додаткового налаштування параметрів.

6.Модель cross-encoder (BAAI/bge-reranker-v2-m3) застосовується лише до 20 кандидатів через її високу обчислювальну складність та неможливість попереднього індексування. На відміну від bi-encoder, який кодує запит і документи окремо (що дозволяє зберегти ембединги фрагментів заздалегідь), cross-encoder обробляє пару «запит + фрагмент» спільно через механізм самоуваги між усіма токенами обох текстів. Оскільки запит користувача стає відомим лише в момент звернення до системи, обчислення доводиться виконувати в реальному часі, що потребує значних витрат часу та ресурсів GPU. Тому використовується двоетапна схема: швидкий гібридний пошук звужує простір пошуку до 20 найбільш імовірних кандидатів, а точний cross-encoder переранжовує лише цю обмежену вибірку.

Оскільки для оцінки кожної пари «запит — фрагмент» потрібен окремий прямий прохід нейромережі, для переранжування всього поточного корпусу знадобилося б 385 проходів моделі на кожне окреме запитання (за кількістю всіх фрагментів у базі) замість 20, що сповільнило б пошук майже у 20 разів і зробило б систему непрактичною для великих обсягів документів.

7.До контрольного набору BENCHMARK було додано три власні запитання: нове питання з відповіддю в документах ("Чи можна платити за гуртожиток щомісяця?"), перефразування наявного запитання про відвідувачів ("До котрої години дозволено залишатися відвідувачам у гуртожитку?") та питання без відповіді в корпусі ("Який розмір штрафу за втрату студентського квитка?"). Загальна кількість питань зросла з 26 до 29, а вибірка для оцінювання ретриву (answerable) збільшилася з 23 до 25 запитань.

Додавання питання без відповіді не вплинуло на метрики пошуку (hit@1, hit@5, MRR), оскільки такі приклади виключаються з розрахунку ретриву (if gold) і використовуються лише для оцінки коректної відмови на етапі генерації. Водночас додавання нового та перефразованого питань по-різному позначилося на роботі алгоритмів:

Векторний пошук (vector_search) покращив свої показники: hit@1 зріс з 0.61 до 0.64, hit@5 з 0.87 до 0.88, а MRR з 0.74 до 0.76. Нейромережева модель ембедингів успішно зрозуміла семантику перефразованого запитання та знайшла потрібні уривки на перших позиціях.

Лексичний пошук (bm25_search) очікувано погіршився: hit@1 знизився з 0.52 до 0.48, hit@5 з 0.65 до 0.64, а MRR з 0.59 до 0.56. Через синонімічну заміну слів у перефразованому запитанні ("відвідувачам" замість "гості") BM25 не знайшов прямих лексичних збігів.

Гібридний пошук (hybrid_search) показав змішаний результат: hit@5 зріс з 0.91 до 0.92, але hit@1 знизився з 0.61 до 0.56, а MRR з 0.76 до 0.72, оскільки низький ранг перефразованого питання у списку BM25 знизив його підсумкову позицію після об'єднання RRF.

Переранжування (rerank_search) зберегло лідерство: показники залишилися стабільно високими (hit@1 = 0.80, hit@5 = 0.96, MRR = 0.88), підтвердивши, що cross-encoder найточніше оцінює змістову відповідність запиту й фрагмента навіть за нестандартного формулювання.

8.Порівняння метрик ретриву на контрольному наборі з 23 запитань у розділі 7 демонструє чітку градацію ефективності методів пошуку:

Лексичний пошук (bm25_search) має найнижчі показники (hit@1 = 0.52, hit@5 = 0.65, MRR = 0.59), оскільки українська мова характеризується складною морфологією з великою кількістю відмінкових закінчень, а питання користувачів рідко повторюють точні формулювання нормативних документів.

Семантичний пошук (vector_search) суттєво переважає BM25 за повнотою у топ-5 (hit@5 = 0.87 проти 0.65, MRR = 0.74), оскільки ембединги вловлюють синоніми та загальний зміст, хоча на першу позицію правильний фрагмент потрапляє лише у 61% випадків (hit@1 = 0.61).

Гібридний пошук (hybrid_search) об'єднує переваги обох підходів за допомогою RRF, що підвищує hit@5 до 0.91 та MRR до 0.76 (зокрема у запитах із рідкісними термінами та абревіатурами), але показник hit@1 залишається на рівні 0.61.

Переранжування (rerank_search) забезпечує найкращі результати за всіма метриками (hit@1 = 0.83, hit@5 = 0.96, MRR = 0.90).

Найсильніший вплив переранжування саме на метрику hit@1 (стрибок на 22 відсоткові пункти — з 0.61 до 0.83, тоді як hit@5 зростає лише з 0.91 до 0.96) пояснюється архітектурною особливістю cross-encoder. На етапі гібридного пошуку правильний фрагмент у 91% випадків уже потрапляє до першої п'ятірки, проте часто посідає 2–5 місця, поступаючись сусіднім загальним пунктам того самого документа. Cross-encoder обробляє запит і текст фрагмента спільно, аналізуючи прямі зв'язки між усіма їхніми токенами, завдяки чому точно визначає уривок із конкретною відповіддю та переміщує його з нижчих позицій топ-20 безпосередньо на перше місце.

9.Використання однієї й тієї самої невеликої моделі (4 млрд параметрів без режиму міркувань) як генератора і судді створює три проблеми: самоупередженість (модель схильна схвалювати власний стиль тексту), спільні помилки розуміння складних правил документа та надмірний формалізм. Оскільки суддя відповідає лише "так" або "ні" без попереднього аналізу, він може сприйняти коректне доповнення до відповіді як суперечність короткому еталону.

Ручна перевірка п'яти прикладів із таблиці evaluation показала збіг із вердиктами судді у 4 із 5 випадків:

Приклад 17 (стипендія ректора): суддя поставив False, хоча відповідь системи правильна (вона повністю містить еталон і додає уточнення з документа про переможців олімпіад та конкурсів). Це хибнонегативна помилка судді.

Приклад 23 (шифр спеціальності): суддя справедливо поставив False, оскільки система назвала старий шифр 121 замість чинного F2.

Приклади 1, 2, 3 (оплата гуртожитку за 1-й і 2-й семестри, режим тиші): вердикти судді True повністю збігаються з ручною оцінкою, факти передано точно.


10.У розділі 10 «Аналіз відмов» було виявлено дві проблемні відповіді, причому в обох випадках етап пошуку спрацював успішно (потрібний документ потрапив до переліку знайдених), але природа збоїв виявилася різною:

Питання «Для кого встановлюється стипендія ректора?» — помилка судді.
Ретрив коректно вивів правильний документ 344 на перше місце, посилання у відповіді вказано правильно (цитування: True), а сама генеративна модель надала повну й достовірну відповідь на основі документа (зазначивши студентів денної форми навчання з суттєвими досягненнями та переможців олімпіад і конкурсів). Проте модель-суддя помилково виставила оцінку False, оскільки розширена відповідь містила більше деталей із тексту документа, ніж лаконічний еталон.

Питання «Який шифр спеціальності "Інженерія програмного забезпечення" в чинній освітній програмі?» — помилка генерації (спричинена суперечливими джерелами).
Попри те що чинний документ 3320 був присутній серед знайдених фрагментів на третій позиції, інші чотири позиції в контексті зайняли фрагменти застарілої редакції програми 2024 року (3329). Мовна модель не змогла коректно розмежувати чинну та архівну версії документа, обрала старий шифр 121 із другого фрагмента замість актуального F2 та послалася на нечинне джерело (правильно: False, цитування: False).